# Session 14 · Case study: LLM versus trained classifier on 200 decisions

**Block 3 practice.** Compare a generative model with the trained TF-IDF classifier on the same 200 decisions: quality, cost, latency and data protection. A prompt cannot list all 1,229 headings, so the LLM chooses among the **ten candidate headings** proposed by the classifier, with structured output.

> **Model server.** Cells marked `REQUIRES A MODEL SERVER` call a language model through an OpenAI-compatible API. Default: a local Ollama server (`ollama pull qwen2.5:3b`, then `ollama serve`). For a hosted provider set `LLM_BASE_URL`, `LLM_MODEL` and `LLM_API_KEY` as environment variables before starting Jupyter; never write a key into the notebook. **Without a server the notebook still runs:** the LLM cells are skipped and the table shows "not run".

Theory: [03-generative-models-through-an-api.md](../theory/03-generative-models-through-an-api.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
print(decisions.shape)

## 1. The trained classifier, the 200 evaluation decisions and their candidates

In [ ]:
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline

year = decisions["start_date"].dt.year
train, valid = decisions[year <= 2021], decisions[year >= 2022]
tfidf_clf = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True),
                          SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None, random_state=0, n_jobs=-1))
tfidf_clf.fit(train["description"], train["heading"])

evalset = valid.sample(200, random_state=0)           # the same 200 decisions for every method
t0 = time.perf_counter()
scores = tfidf_clf.decision_function(evalset["description"])
tf_sec = (time.perf_counter() - t0) / len(evalset)
top10 = tfidf_clf.classes_[np.argsort(-scores, axis=1)[:, :10]]
tf_pred = pd.Series(top10[:, 0], index=evalset.index)
recall = {k: np.mean([h in row[:k] for h, row in zip(evalset["heading"], top10)]) for k in [1, 3, 5, 10]}
print("candidate recall (true heading among the top k):", recall)
print(evalset["language"].value_counts().head(5).to_dict())

In [ ]:
def bootstrap_acc(y_true, y_pred, n_boot=2000, seed=0):
    """Accuracy with a 95 % bootstrap interval (Session 7)."""
    rng = np.random.default_rng(seed)
    correct = (np.asarray(y_true) == np.asarray(y_pred)).astype(float)
    boots = [correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(n_boot)]
    return correct.mean(), *np.percentile(boots, [2.5, 97.5])


print("TF-IDF: accuracy %.2f [%.2f, %.2f]" % bootstrap_acc(evalset["heading"], tf_pred))

## 2. Connect to the model server

In [ ]:
import os

import httpx
from openai import OpenAI

BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
API_KEY = os.getenv("LLM_API_KEY", "ollama")       # Ollama ignores the key
LOCAL = BASE_URL.startswith(("http://localhost", "http://127.0.0.1"))
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60)

try:                                              # is a server reachable?
    r = httpx.get(BASE_URL.rstrip("/") + "/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=3)
    LLM_AVAILABLE = r.status_code == 200
except httpx.HTTPError:
    LLM_AVAILABLE = False
print(BASE_URL, MODEL, "server reachable:", LLM_AVAILABLE)

## 3. Prompt, schema and few-shot examples (runs offline)

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class HeadingChoice(BaseModel):
    heading: str = Field(pattern=r"^\d{4}$")
    confidence: Literal["high", "medium", "low"]


def response_format(candidates):
    schema = HeadingChoice.model_json_schema()
    schema["properties"]["heading"]["enum"] = list(candidates)   # only the candidates are allowed
    return {"type": "json_schema", "json_schema": {"name": "HeadingChoice", "schema": schema}}


SYSTEM = (
    "You are an assistant for EU customs classification. You receive the description of goods of a "
    "Binding Tariff Information request, in any EU language, and a list of candidate HS headings with "
    "their English texts. Choose the single candidate heading that fits the goods best. "
    'Answer only with JSON of the form {"heading": "<four digits>", "confidence": "high" | "medium" | "low"}.'
)
heading_text = nomenclature["heading_description"]


def user_message(description, candidates):
    lines = "\n".join(f"{h}: {heading_text.get(h, '')[:150]}" for h in candidates)
    return f"Description of goods:\n{description[:3000]}\n\nCandidate headings:\n{lines}"


# few-shot examples: two decisions from the TRAINING years (never from the 200), with their candidates
examples = train[train["description"].str.len() < 300].sample(2, random_state=3)
ex_scores = tfidf_clf.decision_function(examples["description"])
FEW_SHOT = []
for (_, row), s in zip(examples.iterrows(), ex_scores):
    cands = list(tfidf_clf.classes_[np.argsort(-s)[:10]])
    if row["heading"] not in cands:
        cands[-1] = row["heading"]
    FEW_SHOT += [{"role": "user", "content": user_message(row["description"], cands)},
                 {"role": "assistant", "content": HeadingChoice(heading=row["heading"], confidence="high").model_dump_json()}]
print(FEW_SHOT[0]["content"][:400])

## 4. Classify the 200 decisions — REQUIRES A MODEL SERVER

With a local 3B model this takes roughly 2–10 minutes per run on a laptop; results are saved to `llm_results/`, so a rerun loads them instead of calling the model again.

In [ ]:
def classify(description, candidates, few_shot=False):
    messages = ([{"role": "system", "content": SYSTEM}] + (FEW_SHOT if few_shot else [])
                + [{"role": "user", "content": user_message(description, candidates)}])
    t0 = time.perf_counter()
    resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0,
                                          response_format=response_format(candidates))
    sec = time.perf_counter() - t0
    try:
        heading = HeadingChoice.model_validate_json(resp.choices[0].message.content).heading
        if heading not in candidates:
            heading = "invalid"
    except Exception:                       # count invalid answers instead of crashing
        heading = "invalid"
    return {"heading": heading, "sec": sec, "tok_in": resp.usage.prompt_tokens,
            "tok_out": resp.usage.completion_tokens}


results_dir = Path("llm_results")
llm = {}
for mode in ["zero-shot", "few-shot"]:
    path = results_dir / f"{MODEL.replace(':', '_').replace('/', '_')}_{mode}.csv"
    if path.exists():
        llm[mode] = pd.read_csv(path, index_col=0, dtype={"heading": str})
    elif LLM_AVAILABLE:
        rows = [classify(d, list(c), few_shot=(mode == "few-shot")) for d, c in zip(evalset["description"], top10)]
        llm[mode] = pd.DataFrame(rows, index=evalset.index)
        results_dir.mkdir(exist_ok=True)
        llm[mode].to_csv(path)
    else:
        print(f"{mode}: no model server, skipped")
{mode: (df["heading"] == "invalid").sum() for mode, df in llm.items()}

## 5. Comparison table

In [ ]:
PRICE_IN, PRICE_OUT = 0.15, 0.60    # EXAMPLE hosted prices in USD per 1M tokens; check the provider
truth = evalset["heading"]


def chapter_acc(pred):
    return np.mean(pd.Series(pred).astype(str).str[:2].to_numpy() == truth.str[:2].to_numpy())


rows = []
acc, lo, hi = bootstrap_acc(truth, tf_pred)
rows.append({"method": "TF-IDF + linear SVM", "accuracy": acc, "CI_low": lo, "CI_high": hi,
             "chapter_accuracy": chapter_acc(tf_pred), "sec_per_decision": tf_sec, "USD_per_1000": 0.0,
             "invalid": 0, "data_leave_machine": "no"})
rows.append({"method": "upper bound: true heading among 10 candidates", "accuracy": recall[10]})
for mode in ["zero-shot", "few-shot"]:
    if mode not in llm:
        rows.append({"method": f"LLM {MODEL} {mode}", "accuracy": "not run"})
        continue
    df = llm[mode]
    acc, lo, hi = bootstrap_acc(truth, df["heading"])      # invalid answers count as errors
    usd = 0.0 if LOCAL else (df["tok_in"].mean() * PRICE_IN + df["tok_out"].mean() * PRICE_OUT) / 1e6 * 1000
    rows.append({"method": f"LLM {MODEL} {mode}", "accuracy": acc, "CI_low": lo, "CI_high": hi,
                 "chapter_accuracy": chapter_acc(df["heading"]), "sec_per_decision": df["sec"].median(),
                 "USD_per_1000": usd, "invalid": int((df["heading"] == "invalid").sum()),
                 "data_leave_machine": "no" if LOCAL else "yes, to " + BASE_URL})
table = pd.DataFrame(rows).set_index("method")
table

For a local model, `USD_per_1000` is 0 but the hardware time is not: compare `sec_per_decision` × 113,188 test decisions with the TF-IDF model.

## 6. Where do the methods disagree? — REQUIRES RESULTS FROM SECTION 4

In [ ]:
if "few-shot" in llm:
    cmp = pd.DataFrame({"language": evalset["language"], "true": truth, "tfidf": tf_pred,
                        "llm": llm["few-shot"]["heading"], "keywords": evalset["keywords"].str[:60],
                        "description": evalset["description"].str[:100]})
    print(pd.crosstab(cmp["tfidf"] == cmp["true"], cmp["llm"] == cmp["true"],
                      rownames=["tf-idf right"], colnames=["llm right"]))
    display(cmp[cmp["tfidf"] != cmp["llm"]].head(10))
else:
    print("No LLM results: run section 4 with a model server first.")

## Your turn

1. Fill in the table with a local model (and, if your team has access, a hosted model). Do the confidence intervals of the methods overlap?
2. Read ten decisions on which TF-IDF and the LLM disagree. Who is right more often, and in which error categories of Session 13?
3. Try the two-step design: first let the model choose one of the 97 chapters (`nomenclature["chapter_description"]`), then one heading of that chapter. Compare accuracy and tokens with the candidate design.
4. Write a recommendation of five sentences for the head of a customs classification unit: which method would you deploy as a suggestion tool for officers, and why? Address quality, cost, latency and data protection (new BTI requests contain confidential business information).

In [ ]:
# your notes or code here